# Robustness Analysis

This notebook stress-tests the main descriptive result from the primary econometric analysis using the cleaned LGA-level panel.

Main result restated: in the preferred LGA-level specification, the distance-to-CBD coefficient is negative but not statistically significant once crime, schools, and health access are controlled for ($β ≈ -0.82$, $p ≈ 0.16$). The claim being graded is descriptive, not causal.

The checks below ask whether that negative association survives alternative control sets, alternative samples, alternative functional form, and an alternative inference choice.

In [1]:
from collections import OrderedDict
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / 'Data' / 'clean' / 'final_panel.csv').exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError('Could not locate Data/clean/final_panel.csv from the notebook working directory.')

data = pd.read_csv(ROOT / 'Data' / 'clean' / 'final_panel.csv')

lga = (
    data.groupby('lga_name', as_index=False)
    .agg({
        'median_rent': 'mean',
        'straight_line_km': 'mean',
        'offence_count': 'mean',
        'total_schools': 'mean',
        'gp_clinics_per_1000': 'mean',
        'independent_schools': 'mean',
        'allied_health_per_1000': 'mean',
        'dental_sites_per_1000': 'mean',
        'pharmacies_per_1000': 'mean',
        'acsc_per_1000': 'mean',
        'driving_km': 'mean',
        'year': 'count',
    })
    .rename(columns={'year': 'n_obs'})
)

base = lga[['lga_name', 'median_rent', 'straight_line_km', 'offence_count', 'total_schools', 'gp_clinics_per_1000']].dropna().copy()
extended = lga.dropna(subset=[
    'median_rent',
    'straight_line_km',
    'offence_count',
    'total_schools',
    'gp_clinics_per_1000',
    'independent_schools',
    'allied_health_per_1000',
    'dental_sites_per_1000',
    'pharmacies_per_1000',
    'acsc_per_1000',
]).copy()

print(f'Baseline LGA sample size: {len(base)}')
print(f'Extended-control sample size: {len(extended)}')
print('Distance summary for the baseline sample:')
display(base['straight_line_km'].describe())

Baseline LGA sample size: 31
Extended-control sample size: 31
Distance summary for the baseline sample:


count    31.000000
mean     20.356774
std      14.466670
min       0.230000
25%       8.755000
50%      17.510000
75%      28.300000
max      62.110000
Name: straight_line_km, dtype: float64

## Robustness Checks

The main table keeps the preferred specification in column 1 and varies one design choice at a time. The checks cover: controls, sample restriction, functional form, and inference.

Because this is a descriptive claim, the key question is not identification but sensitivity: does the distance gradient survive reasonable specification choices, or is it mostly a by-product of the control set?

In [ ]:
def fit_ols(dataframe, outcome, regressors, log_outcome=False, cov_type=None):
    sample = dataframe[[outcome] + regressors].dropna().copy()
    if log_outcome:
        sample[outcome] = np.log(sample[outcome])
    design = sm.add_constant(sample[regressors])
    model = sm.OLS(sample[outcome], design)
    return model.fit(cov_type=cov_type) if cov_type else model.fit()

def summarize(result, note):
    return {
        'N': int(result.nobs),
        'Distance coef': result.params['straight_line_km'],
        'Std. error': result.bse['straight_line_km'],
        'p-value': result.pvalues['straight_line_km'],
        'R^2': result.rsquared,
        'What varies': note,
    }

q1, q3 = base['straight_line_km'].quantile([0.25, 0.75])
trimmed = base[(base['straight_line_km'] >= q1) & (base['straight_line_km'] <= q3)].copy()
drop_extremes = base[(base['straight_line_km'] > base['straight_line_km'].min()) & (base['straight_line_km'] < base['straight_line_km'].max())].copy()
quadratic = base.copy()
quadratic['distance_sq'] = quadratic['straight_line_km'] ** 2

results = OrderedDict()
results['Main specification'] = summarize(
    fit_ols(base, 'median_rent', ['straight_line_km', 'offence_count', 'total_schools', 'gp_clinics_per_1000']),
    'Preferred controls: crime, schools, and health access.'
)
results['No controls'] = summarize(
    fit_ols(base, 'median_rent', ['straight_line_km']),
    'Removes all controls.'
)
results['Extended controls'] = summarize(
    fit_ols(extended, 'median_rent', ['straight_line_km', 'offence_count', 'total_schools', 'gp_clinics_per_1000', 'independent_schools', 'allied_health_per_1000', 'dental_sites_per_1000', 'pharmacies_per_1000', 'acsc_per_1000']),
    'Adds school and health subcomponents plus ACSC and pharmacy density.'
)
results['Drop extremes'] = summarize(
    fit_ols(drop_extremes, 'median_rent', ['straight_line_km', 'offence_count', 'total_schools', 'gp_clinics_per_1000']),
    'Drops the nearest and farthest LGAs by distance.'
)
results['Log rent'] = summarize(
    fit_ols(base, 'median_rent', ['straight_line_km', 'offence_count', 'total_schools', 'gp_clinics_per_1000'], log_outcome=True),
    'Uses log median rent instead of rent levels.'
)
results['HC3 SE'] = summarize(
    fit_ols(base, 'median_rent', ['straight_line_km', 'offence_count', 'total_schools', 'gp_clinics_per_1000'], cov_type='HC3'),
    'Keeps the main model but uses heteroskedasticity-robust SEs.'
)

quadratic_result = sm.OLS(
    quadratic['median_rent'],
    sm.add_constant(quadratic[['straight_line_km', 'distance_sq', 'offence_count', 'total_schools', 'gp_clinics_per_1000']]),
).fit()

table = pd.DataFrame(results)
table = table.loc[['N', 'Distance coef', 'Std. error', 'p-value', 'R^2', 'What varies']]
display(table)

print('Quadratic check (supplementary):')
print(f"distance coefficient = {quadratic_result.params['straight_line_km']:.3f}")
print(f"distance-squared p-value = {quadratic_result.pvalues['distance_sq']:.3f}")

AttributeError: The '.style' accessor requires jinja2

## Interpretation

The robustness checks point to a clear pattern. The simple bivariate relationship is negative and statistically significant, but the coefficient shrinks once the preferred controls are introduced. Adding a richer amenity set attenuates the distance coefficient further and removes statistical significance entirely.

The sample checks do not reverse the sign: dropping the most extreme LGAs leaves a negative coefficient, and the log-rent specification stays negative as well. HC3 standard errors do not materially change the conclusion. The quadratic term is not important.

Substantively, that means the distance-rent gradient is real as a descriptive correlation, but it is fragile to control choice. The evidence supports a weak between-LGA association rather than a strong standalone distance effect.

In [ ]:
output_dir = ROOT / 'outputs' / 'robustness_analysis'
output_dir.mkdir(parents=True, exist_ok=True)

export_table = pd.DataFrame(results)
output_path = output_dir / 'robustness_table.csv'
export_table.to_csv(output_path)
display(export_table)
print(f"Robustness table saved to: {output_path}")